# Multiscale Glow for synthetic 3D volumes
Build a two-level `MultiscaleFlow` with `GlowBlock3d`, train a density, inspect its latent hierarchy, interpolate volumes, and reload it. Arrays have shape `(batch, channels, depth, height, width)`.

This is a small continuous-data example. It does not perform medical image registration, handle spatial metadata, or implement a multiview anatomical alignment objective.

## Setup
Install the current checkout with `python -m pip install -e '.[examples]'` from the repository root. These examples exercise recent source APIs, which may not yet be in an older installed release. Select that environment's kernel and run all cells in order.

All observations are synthetic; no download is required. The small training budgets demonstrate the API rather than a converged scientific model. Increase `steps` and model capacity for a longer experiment.

In [ ]:
import torch
import antsnormflows as nf
from matplotlib import pyplot as plt

torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Package:", nf.__version__, "Device:", device)


In [ ]:
shape = (1, 8, 8, 8)
axis = torch.linspace(-1, 1, 8, device=device)
d, h, w = torch.meshgrid(axis, axis, axis, indexing="ij")
grid = torch.stack([d, h, w])[None]
def draw_volumes(batch):
    center = 0.5 * torch.randn(batch, 3, 1, 1, 1, device=device)
    blob = torch.exp(-((grid - center)**2).sum(dim=1, keepdim=True) / 0.4)
    return blob + 0.15 * torch.randn(batch, *shape, device=device)

observations = draw_volumes(16)
plt.imshow(observations[0, 0, 4].cpu(), cmap="gray")
plt.title("Synthetic volume: central slice")
plt.show()


## Construct the latent hierarchy
The flow list is ordered in the **generative** direction. `Squeeze3d.forward` moves channels into spatial dimensions; its inverse packs spatial blocks into channels.

For `(1, 8, 8, 8)` observations, the inverse first produces `(8, 4, 4, 4)`, splits it into two `(4, 4, 4, 4)` tensors, and packs the continuing branch into `(32, 2, 2, 2)`. The base shapes are therefore `(32, 2, 2, 2)` and `(4, 4, 4, 4)`, preserving 512 scalar dimensions in total.

`gen_clamp` is an emergency activation bound. If clipping or non-finite replacement occurs, exact invertibility and the reported Jacobian no longer describe that clipped map. Keep data in a stable range and verify reconstructions; clipping is not a likelihood correction.

In [ ]:
def make_model(checkpoint=True):
    bases = [nf.distributions.DiagGaussian((32, 2, 2, 2)),
             nf.distributions.DiagGaussian((4, 4, 4, 4))]
    levels = []
    for channels_at_level in [32, 8]:
        blocks = [nf.flows.GlowBlock3d(
            channels_at_level, hidden_channels=8, net_actnorm=False,
            s_cap=0.5, conv_s_cap=1.0, actnorm_s_cap=5.0,
            shift_cap=3.0, gen_clamp=1e4,
        ) for _ in range(2)]
        levels.append(blocks + [nf.flows.Squeeze3d()])
    return nf.MultiscaleFlow(
        bases, levels, [nf.flows.Merge()], class_cond=False,
        grad_checkpoint=checkpoint,
    ).to(device)

model = make_model()
# Initialize ActNorm and cache latent shapes before checkpointed training.
with torch.no_grad():
    latents, _ = model.inverse_and_log_det(observations)
print("Latent shapes:", [tuple(z.shape) for z in latents])
assert sum(z[0].numel() for z in latents) == observations[0].numel()


## Train through the checkpoint-capable transform path
In the current implementation, `grad_checkpoint` applies to `inverse_and_log_det` and `forward_and_log_det` while training. `log_prob` and `forward_kld` have a separate path that does not use this option. We explicitly combine base log densities with the inverse log-determinant to train with checkpointing. It trades extra computation for activation memory; no memory saving is claimed for this tiny example.

In [ ]:
def checkpointed_log_prob(model, x):
    latents, inverse_ld = model.inverse_and_log_det(x)
    return inverse_ld + sum(base.log_prob(z) for base, z in zip(model.q0, latents))

with torch.no_grad():
    torch.testing.assert_close(checkpointed_log_prob(model, observations),
                               model.log_prob(observations), atol=2e-3, rtol=2e-4)
optimizer = torch.optim.Adam(model.parameters(), lr=5e-4)
steps = 30
history = []
model.train()
for step in range(steps):
    optimizer.zero_grad(set_to_none=True)
    loss = -checkpointed_log_prob(model, draw_volumes(8)).mean()
    assert torch.isfinite(loss)
    loss.backward()
    optimizer.step()
    history.append(loss.item() / observations[0].numel())
model.eval()
plt.plot(history)
plt.xlabel("Step")
plt.ylabel("NLL per voxel (nats)")
plt.show()


In [ ]:
with torch.no_grad():
    validation = draw_volumes(8)
    latents, inverse_ld = model.inverse_and_log_det(validation)
    reconstructed, forward_ld = model.forward_and_log_det(latents)
    torch.testing.assert_close(reconstructed, validation, atol=2e-4, rtol=2e-4)
    torch.testing.assert_close(inverse_ld + forward_ld,
                               torch.zeros_like(inverse_ld), atol=2e-3, rtol=0)
    samples, log_q = model.sample(4)
    torch.testing.assert_close(log_q, model.log_prob(samples), atol=3e-3, rtol=2e-4)
    print("Validation NLL per voxel:", -model.log_prob(validation).mean().item() / 512)
    print("Maximum reconstruction error:", (reconstructed - validation).abs().max().item())

fig, axes = plt.subplots(1, 3, figsize=(9, 3))
for ax, volume, title in zip(axes, [validation[0], reconstructed[0], samples[0]],
                             ["Observation", "Reconstruction", "Sample"]):
    ax.imshow(volume[0, 4].cpu(), cmap="gray")
    ax.set_title(title)
plt.tight_layout()
plt.show()


## Interpolate across every latent level
Interpolate corresponding tensors from two observations and decode the entire list. These are model-space interpolations, not registered anatomical correspondences.

In [ ]:
with torch.no_grad():
    weights = torch.linspace(0, 1, 5, device=device)
    interpolated = [torch.cat([(1-a)*z[0:1] + a*z[1:2] for a in weights]) for z in latents]
    volumes, _ = model.forward_and_log_det(interpolated)
fig, axes = plt.subplots(1, 5, figsize=(12, 3))
for i, ax in enumerate(axes):
    ax.imshow(volumes[i, 0, 4].cpu(), cmap="gray")
    ax.set_title(f"Weight {weights[i].item():.2f}")
plt.tight_layout()
plt.show()


## Reload and rebuild the shape cache
The latent-shape cache is not in the saved state dictionary. After loading, explicitly call `inverse_and_log_det` on an appropriately shaped batch before `sample`. Calling `log_prob` alone does not populate this cache in the current implementation.

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory

with TemporaryDirectory() as directory:
    path = Path(directory) / "glow_3d.pt"
    model.save(path)
    restored = make_model(checkpoint=False)
    restored.load(path, map_location=device)
    restored.eval()
    with torch.no_grad():
        restored.inverse_and_log_det(validation[:2])
        torch.testing.assert_close(restored.log_prob(validation), model.log_prob(validation))
        restored_samples, _ = restored.sample(4, temperature=0.8)
    assert restored_samples.shape == (4, *shape)
print("Reloading and temperature sampling passed.")
